<a href="https://colab.research.google.com/github/ManeswarSahu11/PyTorch/blob/main/SimpleNet_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#----------------------------
# Model: PVTNet
# Written by: Maneswar Sahu
# Contact: maneswar.sahu2005@gmail.com
#----------------------------

# Dependency
import time
import math
import random
import numpy as np
import pandas as pd
import scipy.io as sio #loads MATLAB .mat files Because:Your dataset likely comes from MATLAB simulations.

import matplotlib.pyplot as plt

!pip install timm thop torchinfo colorama


import torch
import torch.nn as nn # wyhen layer has parameters
import torch.nn.functional as F #simples ops
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ExponentialLR, CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from timm.layers import trunc_normal_

from thop import profile
from torchinfo import summary

from tqdm.notebook import tqdm_notebook
from colorama import Fore, Style

from google.colab import drive
drive.mount('/content/drive')


import warnings
warnings.filterwarnings("ignore")
print(f'{Fore.BLUE}{Style.BRIGHT}- Dependency:{Fore.RESET}', end=' ')
print(f'{Fore.GREEN}Done{Fore.RESET}{Style.RESET_ALL}')

- Dependency: Done


In [ ]:
#-------------
# Random Seed
#-------------

seed = 2025
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [ ]:
#------------------
# Computing Device
#------------------

# Device
device = torch.device("cpu")
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda:0")

print(f'{Fore.BLUE}{Style.BRIGHT}- Compute Device:{Fore.RESET}', end=' ')
print(f'{Fore.GREEN}{str(device).upper()}{Fore.RESET}{Style.RESET_ALL}')

- Compute Device: CPU


In [ ]:
#---------------------
# Important Functions
#---------------------

# Weight Initialization
def weightinit(m):
    """
    Initialize model weights.
    Args:
       m (nn.Module): pytorch layer

    Returns:
        None : return nothing
    """
    # Linear Layer
    if isinstance(m, nn.Linear):
        trunc_normal_(m.weight, std=.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.constant_(m.bias, 0)

    # LayerNorm
    elif isinstance(m, nn.LayerNorm):
        nn.init.constant_(m.bias, 0)
        nn.init.constant_(m.weight, 1.0)

    # BatchNorm
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.bias, 0)
        nn.init.constant_(m.weight, 1.0)

    # Conv2d
    elif isinstance(m, nn.Conv2d):
        fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
        fan_out //= m.groups
        m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
        if m.bias is not None:
            m.bias.data.zero_()

    # ConvTrnaspose2d
    elif isinstance(m, nn.ConvTranspose2d):
        fan_in = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
        fan_in //= m.groups
        m.weight.data.normal_(0, math.sqrt(2.0 / fan_in))
        if m.bias is not None:
            m.bias.data.zero_()

    return None



# Model Info
def TorchinfoSummary(model, input_data, device):
    """
    Summarize the given PyTorch model
    Args:
        model (nn.Module): pytorch model
        input_data (torch.Tensor): sample input to the model
        device (str): compute device

    Returns:
        None : return nothing
    """
    print(f"{Fore.YELLOW}{summary(model=model, input_data=input_data, verbose=False, device=device)}")

    return None



# Thop Params and Flops Calculation
def ThopParamsFlops(model, input_data, device):
    """
    Calculate the parameters and flops of the given PyTorch model
    Args:
        model (nn.Module): pytorch model
        input_data (torch.Tensor): sample input to the model
        device (str): compute device

    Returns:
        None: return nothing
    """
    Flops, Params = profile(model=model, inputs=(input_data.to(device), ), verbose=False)
    print(f"{Fore.MAGENTA}- Toltal FLOPS: {Fore.CYAN}{Flops/1e6}M")
    print(f"{Fore.MAGENTA}- Total Params: {Fore.CYAN}{Params/1e6}M")

    return None



# NMSE Calculation
def NMSE(predicted_channel, actual_channel):
    """
    Calculate the normalized mean square error of the given tensors
    Args:
        predicted_channel (torch.Tensor): reconstructed channel by the model
        actual_channel (torch.Tensor): actual ground truth channel

    Returns:
        float: normalized mean square error
    """
    # Numpy arrays to PyTorch tensor
    if isinstance(actual_channel, np.ndarray):
        actual_channel = torch.tensor(actual_channel, dtype=torch.float32)
    if isinstance(predicted_channel, np.ndarray):
        predicted_channel = torch.tensor(predicted_channel, dtype=torch.float32)

    # Moving to GPU
    actual_channel = actual_channel.to(device)
    predicted_channel = predicted_channel.to(device)

    with torch.no_grad():
        # De-centralize
        actual_channel = actual_channel - 0.5
        predicted_channel = predicted_channel - 0.5
        # NMSE Calculation
        power = actual_channel[:, 0, :, :] ** 2 + actual_channel[:, 1, :, :] ** 2
        difference = actual_channel - predicted_channel
        mse = difference[:, 0, :, :] ** 2 + difference[:, 1, :, :] ** 2
        nmse = 10 * torch.log10((mse.sum(dim=(1, 2)) / power.sum(dim=(1, 2))).mean())

        return float(nmse)



# RHO Calculation
def RHO(actual_channel, predicted_channel):
    """
    Calculate the cosine similarity
    Args:
        actual_channel (torch.Tensor): actual ground truth channel
        predicted_channel (torch.Tensor): reconstructed channel by the model

    Returns:
        float: rho
    """
    actual_channel_real = actual_channel[:, 0, :, :]
    actual_channel_imag = actual_channel[:, 1, :, :]
    actual_channel_comp = (actual_channel_real - 0.5) + 1j * (actual_channel_imag - 0.5)

    predicted_channel_real = predicted_channel[:, 0, :, :]
    predicted_channel_imag = predicted_channel[:, 1, :, :]
    predicted_channel_comp = (predicted_channel_real - 0.5) + 1j * (predicted_channel_imag - 0.5)

    n1 = (torch.sum(torch.conj(actual_channel_comp)*actual_channel_comp, axis=2))
    n1 = n1.type(torch.DoubleTensor)
    n1 = n1.to(device)
    n2 = (torch.sum(torch.conj(predicted_channel_comp)*predicted_channel_comp, axis=2))
    n2 = n2.type(torch.DoubleTensor)
    n2 = n2.to(device)
    aa = torch.square(abs(torch.sum(torch.conj(actual_channel_comp)*predicted_channel_comp, axis=2)))
    rho = torch.mean(torch.mean(aa/(n1*n2), axis=1))

    return rho

In [ ]:
#---------------
# Dataset Class
#---------------

# Dataset Class For Case0 [H1] --> [H1]
class CustomDataset(Dataset):
    def __init__(self, path:str):
        self.mat = sio.loadmat(path)['HT']
        self.x = np.reshape(self.mat, (self.mat.shape[0], 2, 32, 32))

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        samplex = self.x[idx]
        return torch.tensor(samplex, dtype=torch.float32)

In [ ]:
#------------
# DataLoader
#------------

# Dataset
torch.manual_seed(seed)
train_data = CustomDataset("/scratch/work/Yogesh23/Data/Indoor/DATA_Htrainin.mat")
val_data = CustomDataset("/scratch/work/Yogesh23/Data/Indoor/DATA_Hvalin.mat")
test_data = CustomDataset("/scratch/work/Yogesh23/Data/Indoor/DATA_Htestin.mat")

# DataLoader
torch.manual_seed(seed)
train_loader = DataLoader(train_data, batch_size=200, shuffle=True, pin_memory=True, pin_memory_device="cuda:0")
val_loader = DataLoader(val_data, batch_size=200, shuffle=False, pin_memory=True, pin_memory_device="cuda:0")
test_loader = DataLoader(test_data, batch_size=200, shuffle=False, pin_memory=True, pin_memory_device="cuda:0")

print(f'{Fore.BLUE}{Style.BRIGHT}- DataLoader:{Fore.RESET}', end=' ')
print(f'{Fore.GREEN}Created Suceesfully{Fore.RESET}{Style.RESET_ALL}')

FileNotFoundError: [Errno 2] No such file or directory: '/scratch/work/Yogesh23/Data/Indoor/DATA_Htrainin.mat'

In [ ]:
# FFNN
class FFNN(nn.Module):
    """
    input (B, C, H, W)   --> (200, 2, 32, 32)
    output (B, C, H, W)  --> (200, 2, 32, 32)
    """
    def __init__(self, d_model=32, scale=1):
        super(FFNN, self).__init__()
        self.d_model = d_model
        self.scale = scale
        self.fc1 = nn.Linear(d_model, scale*d_model)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(scale*d_model, d_model)

    # forward pass
    def forward(self, x):
        B, C, H, W = x.shape
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)

        return x


# BLOCK
class BLOCK(nn.Module):
    """
    input (B, C, H, W)   --> (200, 2, 32, 32)
    output (B, C, H, W)  --> (200, 2, 32, 32)
    """
    def __init__(self, d_model=32):
        super(BLOCK, self).__init__()
        self.d_model = d_model
        self.norm = nn.LayerNorm(d_model)
        self.ffnn = FFNN(d_model=d_model)

    # forward pass
    def forward(self, x):
        B, C, H, W = x.shape
        x = x + self.ffnn(self.norm(x))

        return x


# ENCODER
class ENCODER(nn.Module):
    """
    input (B, C, H, W)   --> (200, 2, 32, 32)
    output (B, Codeword)  --> (200, 512)
    """
    def __init__(self, d_model=32, codeword=512):
        super(ENCODER, self).__init__()
        self.d_model = d_model
        self.codeword = codeword
        self.feature = nn.Sequential(BLOCK(d_model=d_model),
                                     BLOCK(d_model=d_model))
        self.proj = nn.Linear(in_features=2048, out_features=codeword)

    # forward pass
    def forward(self, x):
        B, C, H, W = x.shape
        x = self.feature(x)
        x = self.proj(x.reshape(B, 2048))

        return x


# DECODER
class DECODER(nn.Module):
    """
    input (B, Codeword)   --> (200, 512)
    output (B, C, H, W)  --> (200, 2, 32, 32)
    """
    def __init__(self, d_model=32, codeword=512):
        super(DECODER, self).__init__()
        self.d_model = d_model
        self.codeword = codeword
        self.proj = nn.Linear(in_features=codeword, out_features=2048)
        self.feature = nn.Sequential(BLOCK(d_model=d_model),
                                     BLOCK(d_model=d_model),
                                     BLOCK(d_model=d_model),
                                     BLOCK(d_model=d_model),
                                     BLOCK(d_model=d_model),
                                     BLOCK(d_model=d_model))

    # forward pass
    def forward(self, x):
        B, Codeword = x.shape
        x = self.proj(x)
        x = self.feature(x.reshape(B, 2, 32, 32))

        return x



# MODEL
class MODEL(nn.Module):
    """
    input (B, Codeword)   --> (200, 512)
    output (B, C, H, W)  --> (200, 2, 32, 32)
    """
    def __init__(self, d_model=32, codeword=512):
        super(MODEL, self).__init__()
        self.d_model = d_model
        self.codeword = codeword
        self.encoder = ENCODER(d_model=d_model, codeword=codeword)
        self.decoder = DECODER(d_model=d_model, codeword=codeword)
        self.outnorm = nn.LayerNorm(d_model)
        self.act = nn.Sigmoid()

    # forward pass
    def forward(self, x):
        B, C, H, W = x.shape
        x = self.encoder(x)
        x = self.decoder(x)
        x = self.outnorm(x)
        x = self.act(x)

        return x


In [ ]:

# Model
torch.manual_seed(seed)
model = MODEL(d_model=32, codeword=512).to(device)

# Loss
criterion = nn.MSELoss().to(device)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-3, betas=(0.9, 0.999), weight_decay=0.01)

# Scheduler
scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=1000, eta_min=5e-5, last_epoch=-1)

In [ ]:
# Model info
input_data = torch.randn(1, 2, 32, 32)
TorchinfoSummary(model=model, input_data=input_data, device=device)

In [ ]:
# Model params and flops
input_data = torch.randn(1, 2, 32, 32)
ThopParamsFlops(model=model, input_data=input_data, device=device)

In [ ]:

# Model Training and Validation
num_epochs = 1000
train_losses = []
val_losses = []
nmse_score = []

for epoch in tqdm_notebook(range(num_epochs), desc="Model Training", colour="#b53fd3", leave=True):
    # Model Training
    model.train()
    train_loss = 0
    for data in tqdm_notebook(train_loader, "Mini Batch Training", colour="#0099ff", leave=False):
        x = data.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, x)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)

    # Scheduler
    scheduler.step()

    if (epoch+1)%1==0:
        # Model Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data in tqdm_notebook(val_loader, "Validating The Model", colour="#0099ff", leave=False):
                x = data.to(device)
                output = model(x)
                loss = criterion(output, x)
                val_loss += loss.item() * x.size(0)

        # Model Testing
        model.eval()
        nmse_error = 0
        with torch.no_grad():
            for data in tqdm_notebook(test_loader, "Calculating NMSE", colour="#0099ff", leave=False):
                x = data.to(device)
                output = model(x)
                nmse = NMSE(output, x)
                nmse_error+=nmse

        # Training Loss
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)
        # Validation Loss
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)
        # NMSE Score
        avg_nmse = nmse_error/len(test_loader)
        nmse_score.append(avg_nmse)

        # Printing Training Details
        print(Fore.LIGHTBLUE_EX + f"- Epoch: {epoch+1}/{num_epochs}" + Style.RESET_ALL)
        print(Fore.RED + f"- Train Loss: {train_loss:.9f} | Validation Loss: {val_loss:.9f}" + Style.RESET_ALL, end=' ')
        print(Fore.CYAN + f"| Current Learning Rate: {scheduler.optimizer.param_groups[0]['lr']:.7f}" + Style.RESET_ALL, end=' ')
        print(Fore.GREEN + f"| NMSE: {avg_nmse:.7f}" + Style.RESET_ALL)

        # Saving Models Checkpoint
        if avg_nmse<=min(nmse_score):
            params = {'epoch': epoch+1,
                      'model': model.state_dict(),
                      'optimizer': optimizer.state_dict(),
                      'scheduler': scheduler.state_dict()}

            # Saving Trained Weight
            if (epoch+1)<=400:
                torch.save(params, f'model400.pth')
            else:
                torch.save(params, f'model1000.pth')

    # Saving Training Details
    np.savetxt(f'nmse_scores.csv', np.array(nmse_score), delimiter=",")
    np.savetxt(f'train_losses.csv', np.array(train_losses), delimiter=",")
    np.savetxt(f'val_losses.csv', np.array(val_losses), delimiter=",")